In [7]:
from pathlib import Path
from dotenv import load_dotenv

# env_path = Path(r"C:\2027\agenticai\02-langchain\.env")
load_dotenv()
# print("Loaded", env_path)

True

In [2]:
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader

docs = DirectoryLoader(
    "hybrid-search-docs",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).load()

print(f"Loaded {len(docs)} documents")
for d in docs:
    print("-", Path(d.metadata["source"]).name)

C:\Users\Training\AppData\Local\Temp\ipykernel_17860\1560457573.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 8 documents
- annual_leave_policy.txt
- aurora_pro_warranty.txt
- error_code_7742.txt
- general_warranty_policy.txt
- holiday_party.txt
- returns_and_exchanges.txt
- sku_zx9001_warranty.txt
- wellness_room.txt


In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

## chroma in memory, not a server instance
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="hybrid_lab",
)

# k=3, not 1. Hybrid fusion needs a short list from each retriever.
# With k=1 the two lists often hold different documents and the ranks tie.
similarity_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print("✅ Similarity retriever ready (k=3)")

✅ Similarity retriever ready (k=3)


In [10]:
from pathlib import Path

SKU_QUERY = "What is the warranty period for sku-zx9001?"
SKU_GOLD = "sku_zx9001_warranty.txt"

LEAVE_QUERY = "How many vacation days can an employee take for a holiday?"
LEAVE_GOLD = "annual_leave_policy.txt"

def source_name(doc):
    return Path(doc.metadata["source"]).name

def show(title, query, hits):
    print("\n" + title)
    print("Query:", query)
    for i, doc in enumerate(hits, 1):
        print(f"  {i}. {source_name(doc)}")

show("SIMILARITY — exact token", SKU_QUERY, similarity_retriever.invoke(SKU_QUERY))
sku_top = source_name(similarity_retriever.invoke(SKU_QUERY)[0])
if sku_top != SKU_GOLD:
    print("❌ Similarity did not return the SKU ledger. Top-1 is the 12-month coverage page.")
else:
    print("⚠️ Unexpected — similarity ranked the SKU ledger first.")

show("SIMILARITY — paraphrase", LEAVE_QUERY, similarity_retriever.invoke(LEAVE_QUERY))
leave_top = source_name(similarity_retriever.invoke(LEAVE_QUERY)[0])
if leave_top == LEAVE_GOLD:
    print("✅ Similarity found annual leave from the words vacation / employee / holiday.")
else:
    print("❌ Unexpected top-1:", leave_top)


SIMILARITY — exact token
Query: What is the warranty period for sku-zx9001?
  1. general_warranty_policy.txt
  2. sku_zx9001_warranty.txt
  3. aurora_pro_warranty.txt
❌ Similarity did not return the SKU ledger. Top-1 is the 12-month coverage page.

SIMILARITY — paraphrase
Query: How many vacation days can an employee take for a holiday?
  1. annual_leave_policy.txt
  2. wellness_room.txt
  3. holiday_party.txt
✅ Similarity found annual leave from the words vacation / employee / holiday.


In [ ]:
import re
from langchain_community.retrievers import BM25Retriever

def tokenize(text: str) -> list[str]:
    """Keep hyphenated IDs like sku-zx9001 intact."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

# passing all the docs here
bm25_retriever = BM25Retriever.from_documents(docs, preprocess_func=tokenize)
bm25_retriever.k = 3

show("BM25 — exact token", SKU_QUERY, bm25_retriever.invoke(SKU_QUERY))
if source_name(bm25_retriever.invoke(SKU_QUERY)[0]) == SKU_GOLD:
    print("✅ BM25 returned the SKU ledger. Keyword search finds SKU-ZX9001.")
else:
    print("❌ Unexpected — check hybrid-search-docs/.")

show("BM25 — paraphrase", LEAVE_QUERY, bm25_retriever.invoke(LEAVE_QUERY))
bm25_leave_top = source_name(bm25_retriever.invoke(LEAVE_QUERY)[0])
if bm25_leave_top != LEAVE_GOLD:
    print("❌ BM25 did not return the leave policy. Top-1 is the holiday party note.")
else:
    print("⚠️ Unexpected — BM25 ranked annual leave first.")


BM25 — exact token
Query: What is the warranty period for sku-zx9001?
  1. sku_zx9001_warranty.txt
  2. holiday_party.txt
  3. returns_and_exchanges.txt
✅ BM25 returned the SKU ledger. Keyword search finds SKU-ZX9001.

BM25 — paraphrase
Query: How many vacation days can an employee take for a holiday?
  1. holiday_party.txt
  2. annual_leave_policy.txt
  3. sku_zx9001_warranty.txt
❌ BM25 did not return the leave policy. Top-1 is the holiday party note.


In [12]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, similarity_retriever],
    weights=[0.5, 0.5],
)

show("HYBRID — exact token", SKU_QUERY, hybrid_retriever.invoke(SKU_QUERY))
show("HYBRID — paraphrase", LEAVE_QUERY, hybrid_retriever.invoke(LEAVE_QUERY))

sku_ok = source_name(hybrid_retriever.invoke(SKU_QUERY)[0]) == SKU_GOLD
leave_ok = source_name(hybrid_retriever.invoke(LEAVE_QUERY)[0]) == LEAVE_GOLD
print("✅ Hybrid recovered both gold documents." if sku_ok and leave_ok else "❌ Hybrid missed a gold document.")


HYBRID — exact token
Query: What is the warranty period for sku-zx9001?
  1. sku_zx9001_warranty.txt
  2. general_warranty_policy.txt
  3. holiday_party.txt
  4. returns_and_exchanges.txt
  5. aurora_pro_warranty.txt

HYBRID — paraphrase
Query: How many vacation days can an employee take for a holiday?
  1. annual_leave_policy.txt
  2. holiday_party.txt
  3. wellness_room.txt
  4. sku_zx9001_warranty.txt
✅ Hybrid recovered both gold documents.


In [ ]:
cases = [
    (SKU_QUERY, SKU_GOLD),
    (LEAVE_QUERY, LEAVE_GOLD),
]

for query, gold in cases:
    print("=" * 72)
    print("Query:", query)
    print("Gold: ", gold)
    for label, retriever in [
        ("Similarity", similarity_retriever),
        ("BM25      ", bm25_retriever),
        ("Hybrid    ", hybrid_retriever),
    ]:
        top = source_name(retriever.invoke(query)[0])
        mark = "OK" if top == gold else "MISS"
        print(f"  {label} top-1: {top}  [{mark}]")

Query: What is the warranty period for sku-zx9001?
Gold:  sku_zx9001_warranty.txt
  Similarity top-1: general_warranty_policy.txt  [MISS]
  BM25       top-1: sku_zx9001_warranty.txt  [OK]
  Hybrid     top-1: sku_zx9001_warranty.txt  [OK]
Query: How many vacation days can an employee take for a holiday?
Gold:  annual_leave_policy.txt
  Similarity top-1: annual_leave_policy.txt  [OK]
  BM25       top-1: holiday_party.txt  [MISS]
  Hybrid     top-1: annual_leave_policy.txt  [OK]
